# Problema di soddisfacimento vincoli, ricerca in profondità

- **Stato iniziale**: nessuna variabile assegnata
- **Stato finale**: assegnazione completa delle variabili
- **Stato corrente**: assegnazione parziale delle variabili
- **Operatore**: assegnare un valore ad una variabile
- **Cammino**: sequenza di assegnazioni che porta allo stato corrente, ininfluente
- **Costo del cammino**: costante

In [ ]:
adjacency = {
    'A' : ['B', 'G', 'F'],
    'B' : ['A', 'C'],
    'C' : ['B', 'D', 'E', 'F'],
    'D' : ['C'],
    'E' : ['C'],
    'F' : ['A','C','G'],
    'G' : ['A', 'F']
}

colors = ['red', 'green', 'blue', 'yellow']

In [2]:
# Creiamo un agente per risolvere il teorema dei 4 colori

class DFSAgent:
    def __init__(self, adjacency, colors):
        self.colors = colors
        self.adjacency = adjacency
        self.state = {} # state sarà un dizionario {nodo: colore}

    def valid_coloring(self, state, new_node = None):
        for node in state if new_node is None else [new_node]:
            for neighbor in self.adjacency[node]:
                if neighbor in state and state[node]==state[neighbor]:
                    return False
        return True
     # Il controllo dei vincoli può essere limitato SOLO ai vicini del nodo 
     # appena assegnato assumendo che tutti i nodi precedenti siano già stati 
     # assegnati correttamente

    def next_states(self, state):
        for node in self.adjacency:
            if node not in state:
                for color in self.colors:
                    new_state = state.copy()
                    new_state[node] = color
                    if self.valid_coloring(new_state):
                        yield (node,color)

    def dfs(self):
        if len(self.state) == len(self.adjacency):
            yield self.state
        for new_state in self.next_states(self.state):
            self.state = new_state
            yield from self.dfs()

    # l'implementazione ricorsiva ci permette di non dover gestire 
    # manualmente la pila degli stati da esplorare, delegando questa responsabilità
    # allo stack delle chiamate ricorsive 

    # inoltre è più più compatta e leggibile

    


## Versione ottimizzata DFS

le ottimizzazioni che vedremo:

- testare solo un ordine di assegnazione delle variabili, se uno fallisce falliranno anche le altre permutazioni, meglio testare solo un ordine.

- **forward checking**: se qualche variabile non ha più valori assegnabili, è inutile continuare as assegnare le variabili rimanenti.

- **Most constrained variable**: scegliere come prossima variabile da assegnare quella con il minor numero di valori assegnabili. Questa è la variabile che ha la maggior probabilità di causare un fallimento, meglio scoprire subito se c'è un problema.

- **Least costraining value**: scegliere come prossimo valore da assegnare quello che limita meno le scelte future. Questo permette di evitare di bloccarsi in vicoli ciechi. 

In [3]:
class DFSAgentOptim(DFSAgent):
    def __init__(self, adjacency, colors, mrw=False, lcv=False, fwc=False):
        super().__init__(adjacency, colors)
        self.mrw=mrw
        self.lcv=lcv
        self.fwc=fwc

    #STEP 1:
    # come controlliamo i valori rimanenti? Funzione di utilità che restituisce il numero di valori
    # rimanenti dato uno stato ed una variabile
    def legal_values(self,state,node):
        return len(self.colors) - len({state[neighbor] for neighbor in 
                                       # crea un set contenente i colori già usati dai vicini assegnati.
                                       # L'insieme elimina automaticamente i duplicati
                                       self.adjacency[node] if neighbor in state}) # controlla solo i vicini già con un colore assegnato
    

    #STEP 2: restituire TRUE solo se tutte le variabili non assegnate hanno almeno un valore assegnabile
    def forward_checking(self,state):
        return all(self.legal_values(state,node) > 0 for node in 
                   self.adjacency if node not in state) if self.fwc else True
        
    # Most Constrained Variable = Variabile NON ASSEGNATE con il minor numero di valori assegnabili

    def MCV(self, state):
        # key è un parametro che trova il valore che minimizza un criterio arbitrario
        return min([n for n in self.adjacency if n not in state], key=lambda
                    n: self.legal_values(state, n) if self.mrw else 0)


    # LEAST CONSTRAINING VALUE, ci serve una funzione di utilità che conta quante opzioni
    # rimuove l'assegnazione di un valore e di una variabile 
    def removed_neighbors_colors(self, state, node, color):
        neighbors = [n for n in self.adjacency[node] if n not in state]
        return sum(self.legal_values(state, n) for n in neighbors) - sum(self.legal_values({**state, node: color}, n) for n in neighbors)

    # Fatto ciò possiamo usare il metodo sorted per ordinare i colori in ordine crescente di quanti
    # valori rimuovono se assegnati al nodo dato

    def LCV_sort(self,state,node):
        return sorted(self.colors, key=lambda c:
                      self.removed_neighbors_colors(state, node, c) if self.lcv else 0)

    def next_states(self, state):
        node = self.MCV(state) # Prima ottimizzazione, ordine fisso
        return ( new_state
                for color in self.LCV_sort(state, node)
                if self.valid_coloring(new_state := {**state, node: color},
                    new_node=node) and self.forward_checking(new_state))

Queste ottimizzazioni sono delle ottime *euristiche* per ridurre il numero di assegnazioni fallimentari. A seconda della complessità delle verifiche da fare i vantaggi **non sempre** potrebbero giustificare l'overhead computazionale aggiuntivo. 